# Drawdown Interval Risk-State Coverage v1

Purpose: continue from `state_conditioned_multi_objective_bc_screen_v1`, where reentry participation lifted `wf_2023_2024` relative return to the +30pp target but MaxDD failed. This notebook answers the next mechanism question before another training run:

1. Did the fold-local risk-state features cover the actual max-drawdown intervals?
2. If covered, did the actor still carry too much position?
3. If not covered, which short-horizon risk-state feature family should become the next representation / checkpoint-scoring candidate?

Governance:

- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `state_conditioned_multi_objective_bc_screen_v1` and `sequence_trend_joint_warm_start_official_eval_v1`.
- Role: diagnose / model-selection handoff, not promotion.
- Target windows: `wf_2022`, `wf_2023_2024` official warm-start daily paths.
- Acceptance: a risk-state family is useful only if it covers failed drawdown intervals without becoming an always-on gate and exposes whether the remaining failure is representation, actor fit, or checkpoint selection.
- Changed layer: `model_selection`, `risk_state_representation`, `checkpoint_scoring`.
- External reference: Qlib modular workflow (`https://github.com/microsoft/qlib`, `https://arxiv.org/abs/2009.11189`), FinRL risk-aware DRL framing (`https://github.com/AI4Finance-Foundation/FinRL`, `https://arxiv.org/abs/2011.09607`), multi-objective sequential decision trade-offs (`https://arxiv.org/abs/1312.4077`), and BC distribution-shift caution (`https://proceedings.mlr.press/v15/ross11a.html`).
- Closest success: `cap10_sequence_risk_checkpoint_penalty_screen_v1` showed sequence-risk state can capture 2022 breach intervals, but did not solve official promotion.
- Closest failure: `state_conditioned_multi_objective_bc_screen_v1` raised participation but failed MaxDD.
- Why not duplicate: this notebook does not train or sweep floor/cap targets; it diagnoses interval-level signal coverage and position exposure using official warm-start daily paths.


In [1]:

from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import max_drawdown_interval_metrics
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "drawdown_interval_risk_state_coverage_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1"
REFERENCE_CARD = ROOT / "runtime/model_selection/card/sequence_trend_joint_warm_start_official_eval_v1"
HANDOFF_CARD = ROOT / "runtime/model_selection/card/warm_start_official_failure_handoff_v1"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
RISK_CAP_EXPOSURE = 0.12
WALK_FORWARD_WINDOWS = {
    "wf_2022": {"train": ("2010-03-11", "2020-12-31"), "valid": ("2021-01-01", "2021-12-31"), "test": ("2022-01-01", "2022-12-31")},
    "wf_2023_2024": {"train": ("2010-03-11", "2022-12-31"), "valid": ("2023-01-01", "2023-12-31"), "test": ("2024-01-01", "2024-12-31")},
}
FEATURE_COLS = [
    "anchor_gap_180", "anchor_slope_180", "trend_gap_21", "trend_gap_63", "trend_slope_21", "trend_slope_63",
    "mom_21", "mom_63", "realized_vol_21", "realized_vol_63", "vol_of_vol_21", "drawdown_21", "drawdown_63",
    "cum_drop_30", "loss_streak_21", "new_low_ratio_21", "downside_semivol_21", "vix_percentile_252",
    "tracking_error_vol_21", "relative_strength_21", "regime_risk", "regime_risk_no_vix", "regime_risk_velocity_5",
]
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://arxiv.org/abs/1312.4077",
    "https://proceedings.mlr.press/v15/ross11a.html",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def q(frame: pd.DataFrame, col: str, quantile: float) -> float:
    return float(pd.to_numeric(frame[col], errors="coerce").dropna().quantile(float(quantile)))


def build_signal_masks(features: pd.DataFrame, thresholds: dict[str, float]) -> dict[str, pd.Series]:
    # Existing long-horizon mask from the previous state-conditioned target screen.
    existing = (
        ((features["drawdown_63"] <= thresholds["drawdown_63_q20"]) & (features["trend_slope_63"] <= thresholds["trend_slope_63_q55"]))
        | ((features["vix_percentile_252"] >= thresholds["vix_percentile_252_q80"]) & (features["realized_vol_63"] >= thresholds["realized_vol_63_q80"]))
        | ((features["regime_risk"] >= thresholds["regime_risk_q80"]) & (features["mom_63"] <= thresholds["mom_63_q40"]))
        | ((features["regime_risk_no_vix"] >= thresholds["regime_risk_no_vix_q80"]) & (features["tracking_error_vol_21"] >= thresholds["tracking_error_vol_21_q80"]))
    )
    short_drawdown = (
        (features["drawdown_21"] <= thresholds["drawdown_21_q20"])
        | (features["cum_drop_30"] >= thresholds["cum_drop_30_q80"])
        | (features["loss_streak_21"] >= thresholds["loss_streak_21_q80"])
        | (features["new_low_ratio_21"] >= thresholds["new_low_ratio_21_q80"])
    )
    short_volshock = (
        (features["realized_vol_21"] >= thresholds["realized_vol_21_q80"])
        | (features["vol_of_vol_21"] >= thresholds["vol_of_vol_21_q80"])
        | (features["downside_semivol_21"] >= thresholds["downside_semivol_21_q80"])
    )
    trend_rollover = (features["trend_slope_21"] <= thresholds["trend_slope_21_q40"]) & (features["mom_63"] >= thresholds["mom_63_q60"])
    risk_regime = (
        (features["regime_risk"] >= thresholds["regime_risk_q80"])
        | (features["tracking_error_vol_21"] >= thresholds["tracking_error_vol_21_q80"])
        | (features["vix_percentile_252"] >= thresholds["vix_percentile_252_q80"])
    )
    two_of_four = (short_drawdown.astype(int) + short_volshock.astype(int) + trend_rollover.astype(int) + risk_regime.astype(int)) >= 2
    return {
        "existing_long_horizon_risk": existing.fillna(False),
        "short_drawdown_or_newlow": short_drawdown.fillna(False),
        "short_volshock": short_volshock.fillna(False),
        "trend_rollover_after_momentum": trend_rollover.fillna(False),
        "risk_regime_or_tracking_error": risk_regime.fillna(False),
        "short_drawdown_and_vol": (short_drawdown & short_volshock).fillna(False),
        "two_of_four_short_risk": two_of_four.fillna(False),
    }


def thresholds_from_train(train_features: pd.DataFrame) -> dict[str, float]:
    spec = {
        "drawdown_63_q20": ("drawdown_63", 0.20),
        "trend_slope_63_q55": ("trend_slope_63", 0.55),
        "vix_percentile_252_q80": ("vix_percentile_252", 0.80),
        "realized_vol_63_q80": ("realized_vol_63", 0.80),
        "regime_risk_q80": ("regime_risk", 0.80),
        "mom_63_q40": ("mom_63", 0.40),
        "mom_63_q60": ("mom_63", 0.60),
        "regime_risk_no_vix_q80": ("regime_risk_no_vix", 0.80),
        "tracking_error_vol_21_q80": ("tracking_error_vol_21", 0.80),
        "drawdown_21_q20": ("drawdown_21", 0.20),
        "cum_drop_30_q80": ("cum_drop_30", 0.80),
        "loss_streak_21_q80": ("loss_streak_21", 0.80),
        "new_low_ratio_21_q80": ("new_low_ratio_21", 0.80),
        "realized_vol_21_q80": ("realized_vol_21", 0.80),
        "vol_of_vol_21_q80": ("vol_of_vol_21", 0.80),
        "downside_semivol_21_q80": ("downside_semivol_21", 0.80),
        "trend_slope_21_q40": ("trend_slope_21", 0.40),
    }
    return {name: q(train_features, col, quantile) for name, (col, quantile) in spec.items()}

source_selection = load_json(SOURCE_CARD / "selection.json")
source_summary = load_json(SOURCE_CARD / "summary.json")
source_candidate = str(source_selection["selected_candidate"])
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
print("source", source_selection["decision"], source_candidate)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[90354:MainThread](2026-05-26 18:33:42,120) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[90354:MainThread](2026-05-26 18:33:42,872) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[90354:MainThread](2026-05-26 18:33:42,873) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


source state_conditioned_multi_objective_drawdown_failed_no_promotion multi_obj_floor45_cap20_riskfirst


In [2]:

feature_cache: dict[str, pd.DataFrame] = {}
threshold_rows: list[dict[str, Any]] = []
for fold, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    features = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").reset_index()[["datetime", *FEATURE_COLS]].copy()
    features["datetime"] = pd.to_datetime(features["datetime"]).dt.normalize()
    train = features[(features["datetime"] >= pd.Timestamp(spec["train"][0])) & (features["datetime"] <= pd.Timestamp(spec["train"][1]))].copy()
    test = features[(features["datetime"] >= pd.Timestamp(spec["test"][0])) & (features["datetime"] <= pd.Timestamp(spec["test"][1]))].copy()
    thresholds = thresholds_from_train(train)
    masks = build_signal_masks(test, thresholds)
    for name, mask in masks.items():
        test[name] = mask.astype(bool).to_numpy()
    feature_cache[fold] = test
    threshold_rows.append({"fold": fold, **thresholds})

feature_thresholds = pd.DataFrame(threshold_rows)
feature_thresholds.to_csv(CARD / "fold_local_signal_thresholds.csv", index=False)
display(feature_thresholds)


[90354:MainThread](2026-05-26 18:33:45,039) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[90354:MainThread](2026-05-26 18:33:45,040) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[90354:MainThread](2026-05-26 18:33:45,041) INFO - qlib.timer - [log.py:127] - Time cost: 0.005s | Init data Done


[90354:MainThread](2026-05-26 18:33:47,440) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[90354:MainThread](2026-05-26 18:33:47,441) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[90354:MainThread](2026-05-26 18:33:47,442) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


,fold,drawdown_63_q20,trend_slope_63_q55,vix_percentile_252_q80,realized_vol_63_q80,regime_risk_q80,mom_63_q40,mom_63_q60,regime_risk_no_vix_q80,tracking_error_vol_21_q80,drawdown_21_q20,cum_drop_30_q80,loss_streak_21_q80,new_low_ratio_21_q80,realized_vol_21_q80,vol_of_vol_21_q80,downside_semivol_21_q80,trend_slope_21_q40
0,wf_2022,-0.277913,0.061668,0.781746,0.854766,0.634937,0.135672,0.257826,0.619619,0.078065,-0.169820,-0.007717,0.095238,0.190476,0.921593,0.121503,0.640450,0.003829
1,wf_2023_2024,-0.319217,0.057430,0.789683,0.965539,0.645270,0.117794,0.247982,0.628222,0.077580,-0.195128,-0.011675,0.095238,0.190476,1.039652,0.125133,0.749333,-0.000245


In [3]:

candidate_paths: list[dict[str, Any]] = []
source_manifest = pd.read_csv(SOURCE_CARD / "daily_manifest.csv")
for row in source_manifest.to_dict("records"):
    candidate_paths.append({"source": "multi_objective_bc", "candidate": row["candidate"], "fold": row["fold"], "path": ROOT / row["path"]})
for fold in WALK_FORWARD_WINDOWS:
    ref_path = REFERENCE_CARD / f"trend_plain_contract_{fold}_warm_start_official_daily_path.csv"
    if ref_path.exists():
        candidate_paths.append({"source": "trend_plain_reference", "candidate": "trend_plain_contract", "fold": fold, "path": ref_path})

interval_rows: list[dict[str, Any]] = []
coverage_rows: list[dict[str, Any]] = []
feature_snapshot_rows: list[dict[str, Any]] = []
for item in candidate_paths:
    fold = str(item["fold"])
    candidate = str(item["candidate"])
    daily = pd.read_csv(item["path"], parse_dates=["datetime"])
    daily["datetime"] = pd.to_datetime(daily["datetime"]).dt.normalize()
    interval = max_drawdown_interval_metrics(daily)
    interval_mask = (daily["datetime"] >= pd.Timestamp(interval["peak_date"])) & (daily["datetime"] <= pd.Timestamp(interval["trough_date"]))
    features = feature_cache[fold]
    merged = daily.merge(features, on="datetime", how="left")
    interval_merged = merged[interval_mask.to_numpy()].copy()
    interval_rows.append({
        "source": item["source"],
        "candidate": candidate,
        "fold": fold,
        "daily_path": str(item["path"].relative_to(ROOT)),
        **interval,
        "full_gate_drawdown_gap": max(0.0, float(interval["max_dd"]) - TARGET_MAX_DD),
    })
    for signal in [c for c in features.columns if c not in ["datetime", *FEATURE_COLS]]:
        signal_series = merged[signal].fillna(False).astype(bool)
        interval_signal = interval_merged[signal].fillna(False).astype(bool)
        signal_interval_positions = interval_merged.loc[interval_signal, "position"]
        high_position_mask = interval_signal & (pd.to_numeric(interval_merged["position"], errors="coerce").fillna(0.0) > RISK_CAP_EXPOSURE)
        first_signal_date = ""
        if interval_signal.any():
            first_signal_date = str(pd.Timestamp(interval_merged.loc[interval_signal, "datetime"].iloc[0]).date())
        coverage_rows.append({
            "source": item["source"],
            "candidate": candidate,
            "fold": fold,
            "signal_family": signal,
            "overall_signal_share": float(signal_series.mean()),
            "interval_signal_share": float(interval_signal.mean()) if len(interval_signal) else 0.0,
            "interval_signal_days": int(interval_signal.sum()),
            "interval_days": int(len(interval_merged)),
            "interval_avg_pos": float(pd.to_numeric(interval_merged["position"], errors="coerce").fillna(0.0).mean()) if len(interval_merged) else 0.0,
            "signal_interval_avg_pos": float(signal_interval_positions.mean()) if len(signal_interval_positions) else 0.0,
            "signal_interval_high_position_days": int(high_position_mask.sum()),
            "first_interval_signal_date": first_signal_date,
            "max_dd": float(interval["max_dd"]),
        })
    top_features = ["drawdown_21", "drawdown_63", "cum_drop_30", "mom_63", "trend_slope_21", "realized_vol_21", "vol_of_vol_21", "vix_percentile_252", "regime_risk", "tracking_error_vol_21"]
    if not interval_merged.empty:
        feature_snapshot = {f"{col}_interval_mean": float(pd.to_numeric(interval_merged[col], errors="coerce").mean()) for col in top_features}
        feature_snapshot.update({f"{col}_interval_min": float(pd.to_numeric(interval_merged[col], errors="coerce").min()) for col in ["drawdown_21", "drawdown_63"]})
        feature_snapshot.update({f"{col}_interval_max": float(pd.to_numeric(interval_merged[col], errors="coerce").max()) for col in ["cum_drop_30", "realized_vol_21", "vol_of_vol_21", "vix_percentile_252", "regime_risk"]})
        feature_snapshot_rows.append({"source": item["source"], "candidate": candidate, "fold": fold, **feature_snapshot})

interval_metrics = pd.DataFrame(interval_rows).sort_values(["fold", "source", "candidate"]).reset_index(drop=True)
signal_coverage = pd.DataFrame(coverage_rows).sort_values(["fold", "candidate", "signal_family"]).reset_index(drop=True)
feature_snapshots = pd.DataFrame(feature_snapshot_rows).sort_values(["fold", "source", "candidate"]).reset_index(drop=True)
interval_metrics.to_csv(CARD / "drawdown_interval_metrics.csv", index=False)
signal_coverage.to_csv(CARD / "risk_signal_interval_coverage.csv", index=False)
feature_snapshots.to_csv(CARD / "drawdown_interval_feature_snapshots.csv", index=False)
display(interval_metrics)
display(signal_coverage[signal_coverage["candidate"].eq(source_candidate)].sort_values(["fold", "signal_family"]))


,source,candidate,fold,daily_path,peak_date,trough_date,max_dd,interval_return,interval_days,interval_avg_pos,interval_min_pos,interval_max_pos,interval_trades,full_gate_drawdown_gap
0,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2022,runtime/model_selection/card/state_conditioned...,2022-01-03,2022-11-09,30.626799,-30.626799,216,0.131131,0.048456,0.450000,28,15.626799
1,multi_objective_bc,multi_obj_floor55_cap25_balanced,wf_2022,runtime/model_selection/card/state_conditioned...,2022-01-03,2022-12-28,31.808525,-31.808525,249,0.162201,0.000000,0.550000,34,16.808525
2,multi_objective_bc,multi_obj_floor65_cap20_weighted,wf_2022,runtime/model_selection/card/state_conditioned...,2022-01-03,2022-10-14,31.567204,-31.567204,198,0.149538,0.044178,0.482504,29,16.567204
3,trend_plain_reference,trend_plain_contract,wf_2022,runtime/model_selection/card/sequence_trend_jo...,2022-01-03,2022-11-03,20.198957,-20.198957,212,0.114577,0.000000,0.457096,28,5.198957
4,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,runtime/model_selection/card/state_conditioned...,2024-03-07,2024-04-19,16.122029,-16.122029,31,0.222731,0.095961,0.512402,6,1.122029
5,multi_objective_bc,multi_obj_floor55_cap25_balanced,wf_2023_2024,runtime/model_selection/card/state_conditioned...,2024-07-10,2024-12-19,19.036155,-19.036155,115,0.121104,0.000000,0.465111,16,4.036155
6,multi_objective_bc,multi_obj_floor65_cap20_weighted,wf_2023_2024,runtime/model_selection/card/state_conditioned...,2024-07-10,2024-12-19,18.949796,-18.949796,115,0.127884,0.000000,0.479761,16,3.949796
7,trend_plain_reference,trend_plain_contract,wf_2023_2024,runtime/model_selection/card/sequence_trend_jo...,2024-07-10,2024-12-19,12.285728,-12.285728,115,0.110407,0.044066,0.214593,15,0.000000


,source,candidate,fold,signal_family,overall_signal_share,interval_signal_share,interval_signal_days,interval_days,interval_avg_pos,signal_interval_avg_pos,signal_interval_high_position_days,first_interval_signal_date,max_dd
0,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2022,existing_long_horizon_risk,0.828000,0.893519,193,216,0.131131,0.116379,90,2022-01-18,30.626799
1,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2022,risk_regime_or_tracking_error,0.700000,0.754630,163,216,0.131131,0.107411,68,2022-01-18,30.626799
2,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2022,short_drawdown_and_vol,0.808000,0.842593,182,216,0.131131,0.116027,75,2022-01-05,30.626799
3,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2022,short_drawdown_or_newlow,0.808000,0.842593,182,216,0.131131,0.116027,75,2022-01-05,30.626799
4,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock,1.000000,1.000000,216,216,0.131131,0.131131,103,2022-01-03,30.626799
5,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2022,trend_rollover_after_momentum,0.024000,0.023148,5,216,0.131131,0.094781,0,2022-01-11,30.626799
6,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2022,two_of_four_short_risk,0.896000,0.935185,202,216,0.131131,0.119437,92,2022-01-05,30.626799
28,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,existing_long_horizon_risk,0.338645,0.032258,1,31,0.222731,0.127749,1,2024-04-19,16.122029
29,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,risk_regime_or_tracking_error,0.478088,0.419355,13,31,0.222731,0.177475,11,2024-03-11,16.122029
30,multi_objective_bc,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_drawdown_and_vol,0.326693,0.322581,10,31,0.222731,0.142697,7,2024-03-11,16.122029


In [4]:

selected_coverage = signal_coverage[
    signal_coverage["candidate"].eq(source_candidate) & signal_coverage["source"].eq("multi_objective_bc")
].copy()
family_rows = []
for signal, group in selected_coverage.groupby("signal_family", sort=True):
    row = {
        "signal_family": signal,
        "fold_count": int(group["fold"].nunique()),
        "mean_interval_signal_share": float(group["interval_signal_share"].mean()),
        "min_interval_signal_share": float(group["interval_signal_share"].min()),
        "max_overall_signal_share": float(group["overall_signal_share"].max()),
        "mean_signal_interval_avg_pos": float(group["signal_interval_avg_pos"].mean()),
        "total_high_position_signal_days": int(group["signal_interval_high_position_days"].sum()),
        "max_max_dd": float(group["max_dd"].max()),
    }
    for fold in WALK_FORWARD_WINDOWS:
        sub = group[group["fold"].eq(fold)]
        if not sub.empty:
            r = sub.iloc[0]
            row[f"{fold}_interval_signal_share"] = float(r["interval_signal_share"])
            row[f"{fold}_signal_interval_avg_pos"] = float(r["signal_interval_avg_pos"])
            row[f"{fold}_high_position_signal_days"] = int(r["signal_interval_high_position_days"])
    family_rows.append(row)

signal_family_summary = pd.DataFrame(family_rows).sort_values(
    ["min_interval_signal_share", "max_overall_signal_share", "signal_family"],
    ascending=[False, True, True],
).reset_index(drop=True)
signal_family_summary.to_csv(CARD / "risk_signal_family_summary.csv", index=False)

selected_existing = selected_coverage[selected_coverage["signal_family"].eq("existing_long_horizon_risk")].set_index("fold")
selected_two_of_four = selected_coverage[selected_coverage["signal_family"].eq("two_of_four_short_risk")].set_index("fold")
wf2022_existing_share = float(selected_existing.loc["wf_2022", "interval_signal_share"])
wf2022_existing_pos = float(selected_existing.loc["wf_2022", "signal_interval_avg_pos"])
wf2024_existing_share = float(selected_existing.loc["wf_2023_2024", "interval_signal_share"])
wf2024_short_share = float(selected_two_of_four.loc["wf_2023_2024", "interval_signal_share"])
wf2024_short_pos = float(selected_two_of_four.loc["wf_2023_2024", "signal_interval_avg_pos"])

if wf2022_existing_share >= 0.75 and wf2024_existing_share < 0.25 and wf2024_short_share >= 0.45:
    decision = "risk_state_representation_split_needed_no_promotion"
    next_mechanism = "add short-horizon drawdown/volshock risk state and use drawdown-interval-aware checkpoint scoring before any new floor/PPO"
elif wf2022_existing_share >= 0.75 and wf2022_existing_pos > RISK_CAP_EXPOSURE:
    decision = "risk_signal_covered_but_position_too_high_no_promotion"
    next_mechanism = "tighten risk-budget/cash-mode checkpoint scoring before adding reentry participation"
else:
    decision = "risk_state_coverage_unclear_no_promotion"
    next_mechanism = "inspect feature deltas and build a smaller risk-state ablation before training"

selected_interval = interval_metrics[(interval_metrics["candidate"].eq(source_candidate)) & (interval_metrics["source"].eq("multi_objective_bc"))]
selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "profile": PROFILE_NAME,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": ["state_conditioned_multi_objective_bc_screen_v1", "sequence_trend_joint_warm_start_official_eval_v1"],
    "closest_historical_success_case": "cap10_sequence_risk_checkpoint_penalty_screen_v1 captured wf_2022 sequence risk but did not produce a promotion candidate",
    "closest_historical_failure_case": "state_conditioned_multi_objective_bc_screen_v1 lifted 2024 relative return but failed MaxDD",
    "why_not_duplicate": "This notebook does interval-level signal coverage and position-exposure diagnosis instead of training or sweeping reentry floors.",
    "changed_layer": ["model_selection", "risk_state_representation", "checkpoint_scoring"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "selected_source_candidate": source_candidate,
    "promotion_allowed": False,
    "training_allowed_from_this_card": False,
    "ppo_screen_allowed_next": False,
    "wf_2022_existing_risk_interval_share": wf2022_existing_share,
    "wf_2022_existing_risk_signal_avg_pos": wf2022_existing_pos,
    "wf_2023_2024_existing_risk_interval_share": wf2024_existing_share,
    "wf_2023_2024_short_risk_interval_share": wf2024_short_share,
    "wf_2023_2024_short_risk_signal_avg_pos": wf2024_short_pos,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": [
        "fold_local_signal_thresholds.csv",
        "drawdown_interval_metrics.csv",
        "risk_signal_interval_coverage.csv",
        "risk_signal_family_summary.csv",
        "drawdown_interval_feature_snapshots.csv",
        "selection.json",
        "summary.json",
    ],
}
summary = {
    **selection,
    "selected_interval_metrics": selected_interval.to_dict("records"),
    "signal_family_summary": signal_family_summary.to_dict("records"),
    "conclusion": [
        f"For {source_candidate} wf_2022, existing long-horizon risk state covers {wf2022_existing_share:.1%} of the max-drawdown interval, but signal-day average position is {wf2022_existing_pos:.3f}; this is a risk-budget/checkpoint-scoring problem more than missing long-horizon features.",
        f"For {source_candidate} wf_2023_2024, existing long-horizon risk state covers only {wf2024_existing_share:.1%} of the max-drawdown interval, while two_of_four_short_risk covers {wf2024_short_share:.1%}; this is a short-horizon risk representation gap.",
        "No promotion, no actor retention, and no longer PPO from this card. The next experiment should add short-horizon drawdown/volshock state to checkpoint scoring or target generation before another training run.",
        f"Next action: {next_mechanism}.",
    ],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
display(signal_family_summary)
selection


,signal_family,fold_count,mean_interval_signal_share,min_interval_signal_share,max_overall_signal_share,mean_signal_interval_avg_pos,total_high_position_signal_days,max_max_dd,wf_2022_interval_signal_share,wf_2022_signal_interval_avg_pos,wf_2022_high_position_signal_days,wf_2023_2024_interval_signal_share,wf_2023_2024_signal_interval_avg_pos,wf_2023_2024_high_position_signal_days
0,short_volshock,2,0.806452,0.612903,1.000000,0.145342,118,30.626799,1.000000,0.131131,103,0.612903,0.159553,15
1,two_of_four_short_risk,2,0.741786,0.548387,0.896000,0.147348,106,30.626799,0.935185,0.119437,92,0.548387,0.175258,14
2,short_drawdown_or_newlow,2,0.647103,0.451613,0.808000,0.152851,85,30.626799,0.842593,0.116027,75,0.451613,0.189676,10
3,risk_regime_or_tracking_error,2,0.586992,0.419355,0.700000,0.142443,79,30.626799,0.754630,0.107411,68,0.419355,0.177475,11
4,short_drawdown_and_vol,2,0.582587,0.322581,0.808000,0.129362,82,30.626799,0.842593,0.116027,75,0.322581,0.142697,7
5,existing_long_horizon_risk,2,0.462888,0.032258,0.828000,0.122064,91,30.626799,0.893519,0.116379,90,0.032258,0.127749,1
6,trend_rollover_after_momentum,2,0.140606,0.023148,0.087649,0.138978,8,30.626799,0.023148,0.094781,0,0.258065,0.183174,8


{'experiment': 'drawdown_interval_risk_state_coverage_v1',
 'created_at': '2026-05-26T10:33:47.634125+00:00',
 'decision': 'risk_state_representation_split_needed_no_promotion',
 'role': 'diagnose',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': ['state_conditioned_multi_objective_bc_screen_v1',
  'sequence_trend_joint_warm_start_official_eval_v1'],
 'closest_historical_success_case': 'cap10_sequence_risk_checkpoint_penalty_screen_v1 captured wf_2022 sequence risk but did not produce a promotion candidate',
 'closest_historical_failure_case': 'state_conditioned_multi_objective_bc_screen_v1 lifted 2024 relative return but failed MaxDD',
 'why_not_duplicate': 'This notebook does interval-level signal coverage and position-exposure diagnosis instead of training or sweeping reentry floors.',
 'changed_layer': ['model_selection',
  'risk_state_representation',
  'checkpoint_scoring'],
 'target_rel

In [5]:

selection = load_json(CARD / "selection.json")
summary = load_json(CARD / "summary.json")
lines = [
    "## Runtime Conclusion",
    "",
    f"Decision: `{selection['decision']}`. Promotion is blocked.",
    "",
    f"Source candidate: `{selection['selected_source_candidate']}`.",
    "",
    "Key findings:",
]
lines.extend(f"- {line}" for line in summary.get("conclusion", []))
lines.extend([
    "",
    "Main sidecars:",
    "- `drawdown_interval_metrics.csv`",
    "- `risk_signal_interval_coverage.csv`",
    "- `risk_signal_family_summary.csv`",
    "- `drawdown_interval_feature_snapshots.csv`",
    "- `selection.json` / `summary.json`",
    "",
    f"Next action: `{selection['next_mechanism']}`.",
])
display(Markdown("\n".join(lines)))


## Runtime Conclusion

Decision: `risk_state_representation_split_needed_no_promotion`. Promotion is blocked.

Source candidate: `multi_obj_floor45_cap20_riskfirst`.

Key findings:
- For multi_obj_floor45_cap20_riskfirst wf_2022, existing long-horizon risk state covers 89.4% of the max-drawdown interval, but signal-day average position is 0.116; this is a risk-budget/checkpoint-scoring problem more than missing long-horizon features.
- For multi_obj_floor45_cap20_riskfirst wf_2023_2024, existing long-horizon risk state covers only 3.2% of the max-drawdown interval, while two_of_four_short_risk covers 54.8%; this is a short-horizon risk representation gap.
- No promotion, no actor retention, and no longer PPO from this card. The next experiment should add short-horizon drawdown/volshock state to checkpoint scoring or target generation before another training run.
- Next action: add short-horizon drawdown/volshock risk state and use drawdown-interval-aware checkpoint scoring before any new floor/PPO.

Main sidecars:
- `drawdown_interval_metrics.csv`
- `risk_signal_interval_coverage.csv`
- `risk_signal_family_summary.csv`
- `drawdown_interval_feature_snapshots.csv`
- `selection.json` / `summary.json`

Next action: `add short-horizon drawdown/volshock risk state and use drawdown-interval-aware checkpoint scoring before any new floor/PPO`.